In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np
import zarr
import tqdm
import time
import copy
import matplotlib.pyplot as plt

In [ ]:
class MRI_Motion_Dataset(Dataset):
    def __init__(self, moving_zarr_path, fixed_zarr_path):
        super().__init__()
        self.moving_zarr_array = zarr.open(moving_zarr_path, mode='r')
        self.fixed_zarr_array = zarr.open(fixed_zarr_path, mode='r')
        assert self.moving_zarr_array.shape[3] == self.fixed_zarr_array.shape[3]
        self.num_images = self.moving_zarr_array.shape[3]
        original_shape = self.fixed_zarr_array.shape[:3] # H, W, D
        self.padded_shape = list(original_shape)
        for i in range(3):
            if self.padded_shape[i] % 4 != 0:
                self.padded_shape[i] = (self.padded_shape[i] // 4 + 1) * 4
        self.input_shape = (self.padded_shape[2], self.padded_shape[0], self.padded_shape[1]) # D, H, W
        print(f"Dataset initialisiert. Original H,W,D: {original_shape}. Padded H,W,D: {self.padded_shape}")
    def __len__(self):
        return self.num_images
    def __getitem__(self, idx):
        moving_np = self.moving_zarr_array[..., idx]
        fixed_np = self.fixed_zarr_array[..., idx]
        moving_vol = torch.from_numpy(moving_np.astype(np.float32)).permute(2,0,1).unsqueeze(0)
        fixed_vol = torch.from_numpy(fixed_np.astype(np.float32)).permute(2,0,1).unsqueeze(0)
        pad_d, pad_h, pad_w = self.padded_shape[2]-moving_vol.shape[1], self.padded_shape[0]-moving_vol.shape[2], self.padded_shape[1]-moving_vol.shape[3]
        padding = (pad_w//2, pad_w-pad_w//2, pad_h//2, pad_h-pad_h//2, pad_d//2, pad_d-pad_d//2)
        return F.pad(moving_vol, padding), F.pad(fixed_vol, padding)

In [ ]:
class SpatialTransformer3D(nn.Module):
    def __init__(self, size):
        super().__init__()
        vectors = [torch.arange(0, s) for s in size]
        grids = torch.meshgrid(vectors, indexing='ij')
        grid = torch.stack(grids)
        self.register_buffer('grid', grid.unsqueeze(0).float(), persistent=False)
    def forward(self, src, flow):
        new_locs = self.grid + flow
        shape = flow.shape[2:]
        for i in range(len(shape)): new_locs[:, i, ...] = 2 * (new_locs[:, i, ...] / (shape[i] - 1) - 0.5)
        new_locs = new_locs.permute(0, 2, 3, 4, 1)[..., [2, 1, 0]]
        return F.grid_sample(src, new_locs, align_corners=True, padding_mode="border")

In [ ]:
class UNet3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 16)
        self.enc2 = self._conv_block(16, 32)
        self.pool = nn.MaxPool3d(2)
        self.bottleneck = self._conv_block(32, 64)
        self.upconv2 = nn.ConvTranspose3d(64, 32, 2, 2)
        self.dec2 = self._conv_block(64, 32)
        self.upconv1 = nn.ConvTranspose3d(32, 16, 2, 2)
        self.dec1 = self._conv_block(32, 16)
        self.final_conv = nn.Conv3d(16, out_channels, 1)
        self.final_conv.weight.data.zero_()
        self.final_conv.bias.data.zero_()
        
    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
    def forward(self, x_fixed, x_moving):
        x = torch.cat([x_fixed, x_moving], dim=1)
        
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        
        # Bottleneck
        b = self.bottleneck(self.pool(e2))
        
        # Decoder
        d2 = self.upconv2(b)
        
        if d2.shape[2:] != e2.shape[2:]:
            d2 = F.interpolate(d2, size=e2.shape[2:], mode='trilinear', align_corners=False)

        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        
        d1 = self.upconv1(d2)

        if d1.shape[2:] != e1.shape[2:]:
            d1 = F.interpolate(d1, size=e1.shape[2:], mode='trilinear', align_corners=False)

        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        
        return self.final_conv(d1)

In [ ]:
class RegistrationModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.registration_net = UNet3D()
        self.spatial_transformer = SpatialTransformer3D(size=input_size)
    def forward(self, moving, fixed):
        displacement_field = self.registration_net(fixed, moving)
        warped_moving = self.spatial_transformer(moving, displacement_field)
        return warped_moving, displacement_field

In [ ]:
class LocalNCCLoss(nn.Module):
    def __init__(self, window_size=9):
        super().__init__()
        self.avg_pool = nn.AvgPool3d(kernel_size=window_size, stride=1, padding=window_size//2)
    def forward(self, img1, img2):
        mu1 = self.avg_pool(img1); mu2 = self.avg_pool(img2)
        mu1_sq, mu2_sq = mu1*mu1, mu2*mu2
        sigma1_sq = self.avg_pool(img1*img1) - mu1_sq
        sigma2_sq = self.avg_pool(img2*img2) - mu2_sq
        covar = self.avg_pool(img1*img2) - mu1*mu2
        ncc = torch.mean((covar**2) / (sigma1_sq * sigma2_sq + 1e-6))
        return 1 - ncc

In [ ]:
def smooth_loss(displacement_field):
    dy = torch.abs(displacement_field[:, :, 1:, :, :] - displacement_field[:, :, :-1, :, :])
    dx = torch.abs(displacement_field[:, :, :, 1:, :] - displacement_field[:, :, :, :-1, :])
    dz = torch.abs(displacement_field[:, :, :, :, 1:] - displacement_field[:, :, :, :, :-1])
    return (torch.mean(dx**2) + torch.mean(dy**2) + torch.mean(dz**2)) / 3.0

In [ ]:
def train_one_epoch(model, loader, optimizer, similarity_loss_fn, reg_weight, device):
    model.train()
    total_loss, total_sim_loss, total_reg_loss = 0, 0, 0
    
    for moving_batch, fixed_batch in tqdm.tqdm(loader, desc="Training"):
        moving_batch, fixed_batch = moving_batch.to(device), fixed_batch.to(device)
        
        warped_image, displacement = model(moving_batch, fixed_batch)
        
        similarity_loss = similarity_loss_fn(warped_image, fixed_batch)
        regularization_loss = smooth_loss(displacement)
        loss = similarity_loss + reg_weight * regularization_loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_sim_loss += similarity_loss.item()
        total_reg_loss += regularization_loss.item()
        
    avg_loss = total_loss / len(loader)
    avg_sim = total_sim_loss / len(loader)
    avg_reg = total_reg_loss / len(loader)
    return avg_loss, avg_sim, avg_reg

In [ ]:
def validate_one_epoch(model, loader, similarity_loss_fn, reg_weight, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for moving_batch, fixed_batch in tqdm.tqdm(loader, desc="Validating"):
            moving_batch, fixed_batch = moving_batch.to(device), fixed_batch.to(device)
            warped_image, displacement = model(moving_batch, fixed_batch)
            similarity_loss = similarity_loss_fn(warped_image, fixed_batch)
            regularization_loss = smooth_loss(displacement)
            loss = similarity_loss + reg_weight * regularization_loss
            total_loss += loss.item()
            
    return total_loss / len(loader)

In [ ]:
MOVING_ZARR_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/DCE_codeset/DCE_codeset/MRI-Datasets/DCE/DCE"
FIXED_ZARR_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/DCE_codeset/MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr"
BEST_MODEL_SAVE_PATH = "./best_registration_model.pth"

# Trainingsparameter
BATCH_SIZE = 1
INITIAL_LEARNING_RATE = 1e-3
NUM_EPOCHS = 150
SMOOTH_REG_WEIGHT = 0.1
VALIDATION_SPLIT = 0.25
RANDOM_STATE = 42

# "Patience" Parameter
PATIENCE_EPOCHS = 10 # Epochen ohne Verbesserung, bevor LR reduziert wird
LR_REDUCTION_FACTOR = 0.5 # Faktor, um den die LR reduziert wird
MIN_LEARNING_RATE = 1e-6 # Training stoppt, wenn LR diesen Wert unterschreitet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Verwende Gerät: {device}")

full_dataset = MRI_Motion_Dataset(moving_zarr_path=MOVING_ZARR_PATH, fixed_zarr_path=FIXED_ZARR_PATH)
indices = list(range(len(full_dataset)))
train_indices, val_indices = train_test_split(indices, test_size=VALIDATION_SPLIT, random_state=RANDOM_STATE)

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Daten aufgeteilt: {len(train_dataset)} Trainingsbilder, {len(val_dataset)} Validierungsbilder.")

input_size = full_dataset.input_shape
model = RegistrationModel(input_size=input_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=INITIAL_LEARNING_RATE)
similarity_loss_fn = LocalNCCLoss().to(device)

train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_counter = 0

print("\n--- Starte Training ---")
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoche {epoch+1}/{NUM_EPOCHS} | Aktuelle LR: {optimizer.param_groups[0]['lr']:.1e}")
    
    avg_train_loss, avg_sim, avg_reg = train_one_epoch(model, train_loader, optimizer, similarity_loss_fn, SMOOTH_REG_WEIGHT, device)
    train_losses.append(avg_train_loss)
    
    avg_val_loss = validate_one_epoch(model, val_loader, similarity_loss_fn, SMOOTH_REG_WEIGHT, device)
    val_losses.append(avg_val_loss)
    
    print(f"  Train Loss: {avg_train_loss:.6f} (NCC: {avg_sim:.6f}, Smooth: {avg_reg:.6f})")
    print(f"  Val Loss:   {avg_val_loss:.6f}")
    
    if avg_val_loss < best_val_loss:
        print(f"  -> Val Loss verbessert von {best_val_loss:.6f} zu {avg_val_loss:.6f}. Speichere Modell...")
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  -> Keine Verbesserung. Patience: {patience_counter}/{PATIENCE_EPOCHS}")
        if patience_counter >= PATIENCE_EPOCHS:
            print(f"!!! Geduldsgrenze erreicht. Lade bestes Modell und reduziere Lernrate. !!!")
            model.load_state_dict(torch.load(BEST_MODEL_SAVE_PATH))
            
            new_lr = optimizer.param_groups[0]['lr'] * LR_REDUCTION_FACTOR
            if new_lr < MIN_LEARNING_RATE:
                print("!!! Lernrate zu gering. Beende Training frühzeitig. !!!")
                break
            
            for param_group in optimizer.param_groups:
                param_group['lr'] = new_lr
            
            patience_counter = 0

print(f"\n--- Training abgeschlossen. Bestes Modell gespeichert unter: {BEST_MODEL_SAVE_PATH} ---")

plt.figure(figsize=(12, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Trainings- und Validierungsverlauf')
plt.xlabel('Epoche')
plt.ylabel('Total Loss')
plt.legend()
plt.grid(True)
plt.show()

/home/shooty/anaconda3/envs/ds-env-01/lib/python3.13/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


Verwende Gerät: cpu
Dataset initialisiert. Original H,W,D: (256, 256, 50). Padded H,W,D: [256, 256, 52]
Daten aufgeteilt: 750 Trainingsbilder, 250 Validierungsbilder.

--- Starte Training ---

Epoche 1/150 | Aktuelle LR: 1.0e-03


Training:   0%|          | 0/750 [00:07<?, ?it/s]


KeyboardInterrupt: 